In [1]:
import torch as t
from torch import Tensor
from jaxtyping import Float

from nnsight import LanguageModel

import sys

sys.path.append("../../../")

from nngine import alter

device = "cuda"

In [2]:
import torch as t
from torch import Tensor
from jaxtyping import Float
from tqdm import tqdm
import numpy as np

from nnsight.models.UnifiedTransformer import UnifiedTransformer

from transformer_lens import HookedTransformer

device = "cuda"

model = UnifiedTransformer(
    'gpt2-small',
    processing=False,
    center_writing_weights=False,
    center_unembed=False,
    fold_ln=False,
    device=device,
)
tokenizer = model.tokenizer

model.set_use_hook_mlp_in(True)
model.set_use_split_qkv_input(True)
model.set_use_attn_result(True)

Loaded pretrained model gpt2-small into HookedTransformer


In [3]:
from ioi_dataset import IOIDataset

N = 25
clean_dataset = IOIDataset(
    prompt_type='mixed',
    N=N,
    tokenizer=model.tokenizer,
    prepend_bos=False,
    seed=1,
    device=device
)
corr_dataset = clean_dataset.gen_flipped_prompts('ABC->XYZ, BAB->XYZ')

In [4]:
def ave_logit_diff(
    logits: Float[Tensor, 'batch seq d_vocab'],
    ioi_dataset: IOIDataset,
    per_prompt: bool = False
):
    '''
        Return average logit difference between correct and incorrect answers
    '''
    # Get logits for indirect objects
    batch_size = logits.size(0)
    io_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.io_tokenIDs[:batch_size]]
    s_logits = logits[range(batch_size), ioi_dataset.word_idx['end'][:batch_size], ioi_dataset.s_tokenIDs[:batch_size]]
    # Get logits for subject
    logit_diff = io_logits - s_logits
    return logit_diff if per_prompt else logit_diff.mean()


with t.no_grad():
    # with model.trace(clean_dataset.toks):
    #     clean_logits = model.output.logits.save()

    # with model.trace(corr_dataset.toks):
    #     corrupt_logits = model.output.logits.save()

    with model.trace(clean_dataset.toks):
        clean_logits = model.output.save()

    with model.trace(corr_dataset.toks):
        corrupt_logits = model.output.save()

clean_logits = clean_logits.value
corrupt_logits = corrupt_logits.value

clean_logit_diff = ave_logit_diff(clean_logits, clean_dataset).item()
corrupt_logit_diff = ave_logit_diff(corrupt_logits, corr_dataset).item()

def ioi_metric(
    logits: Float[Tensor, "batch seq_len d_vocab"],
    corrupted_logit_diff: float = corrupt_logit_diff,
    clean_logit_diff: float = clean_logit_diff,
    ioi_dataset: IOIDataset = clean_dataset
 ):
    patched_logit_diff = ave_logit_diff(logits, ioi_dataset)
    return (patched_logit_diff - corrupted_logit_diff) / (clean_logit_diff - corrupted_logit_diff)

def negative_ioi_metric(logits: Float[Tensor, "batch seq_len d_vocab"]):
    return -ioi_metric(logits)
    
# Get clean and corrupt logit differences
with t.no_grad():
    clean_metric = ioi_metric(clean_logits, corrupt_logit_diff, clean_logit_diff, clean_dataset)
    corrupt_metric = ioi_metric(corrupt_logits, corrupt_logit_diff, clean_logit_diff, corr_dataset)

print(f'Clean direction: {clean_logit_diff}, Corrupt direction: {corrupt_logit_diff}')
print(f'Clean metric: {clean_metric}, Corrupt metric: {corrupt_metric}')

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Clean direction: 2.805180311203003, Corrupt direction: 1.959859013557434
Clean metric: 0.9999999403953552, Corrupt metric: 0.0


In [5]:
nn_model = LanguageModel(
    'openai-community/gpt2',
    device_map="cuda:0",
    dispatch=True,
    tokenizer=model.tokenizer
)

alter(nn_model)

In [6]:
import new_eap

import importlib

importlib.reload(new_eap)

graph = new_eap.EAP({}, components=["head", "mlp"])

In [7]:
graph.run(
    nn_model,
    model.tokenizer,
    clean_dataset.toks,
    corr_dataset.toks,
    batch_size=25,
    metric=ioi_metric,
)

tensor([[[[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          ...,
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00]],

         [[ 2.2598e-05, -3.1469e-05, -2.2741e-05,  ...,  4.8767e-05,
            3.6194e-05, -1.3305e-04],
          [ 1.4147e-06, -1.3144e-06, -7.1698e-07,  ..., -1.2522e-06,
           -5.4353e-06, -4.3268e-08],
          [ 9.0476e-04,  3.7579e-04, -1.2981e-04,  ..., -1.5116e-03,
            7.2179e-04, -1.0327e-03],
          ...,
     

In [8]:
edges = graph.top_edges(n=100, format=True)

head.10.7 -> [0.036] -> head.11.10.q
head.9.9 -> [-0.035] -> head.11.10.q
head.5.5 -> [0.029] -> head.8.6.v
head.9.9 -> [-0.026] -> head.10.7.q
head.5.5 -> [-0.02] -> head.6.9.q
head.9.6 -> [-0.018] -> head.11.10.q
head.4.11 -> [0.016] -> head.6.9.k
head.9.6 -> [-0.013] -> head.10.7.q
head.5.5 -> [0.012] -> head.7.9.v
head.5.5 -> [0.011] -> head.8.10.v
head.8.6 -> [0.01] -> head.9.9.q
head.10.10 -> [-0.01] -> head.11.10.q
head.8.6 -> [0.01] -> head.10.0.q
head.6.9 -> [0.01] -> head.8.6.v
head.10.0 -> [-0.009] -> head.11.10.q
head.4.11 -> [0.009] -> head.5.5.k
head.9.6 -> [-0.009] -> head.10.0.q
head.5.6 -> [0.008] -> head.6.9.k
head.5.5 -> [0.008] -> head.8.10.k
head.2.2 -> [0.008] -> head.6.9.k
head.3.0 -> [0.008] -> head.6.9.q
head.5.5 -> [0.008] -> head.7.1.q
head.8.10 -> [0.007] -> head.10.0.q
head.3.0 -> [0.007] -> head.7.1.q
head.8.10 -> [0.007] -> head.11.2.q
head.9.6 -> [-0.007] -> head.11.2.q
head.10.1 -> [-0.007] -> head.11.10.q
head.3.0 -> [0.006] -> head.8.10.v
head.7.9 -> 